<a href="https://colab.research.google.com/github/Neeti1987/Pathway-GEM-scripts/blob/main/Leucine_pathway_specific_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === STEP 1: Install dependencies ===
!pip install cobra pandas

import cobra
from cobra import Model, Reaction, Metabolite
import pandas as pd
from google.colab import files

# === STEP 2: Upload KO annotation file ===
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# === STEP 3: Parse KO annotations genome-wise ===
genome_kos = {}
with open(filename, "r") as f:
    for line in f:
        if not line.strip(): continue
        parts = line.strip().split("\t")
        if len(parts) < 2: continue
        genome_id, ko = parts[0], parts[1]
        genome_prefix = genome_id.split("_")[0]
        genome_kos.setdefault(genome_prefix, set()).add(ko)

# === STEP 4: Apply KEGG logic definition for M00432 (Leucine biosynthesis) ===
def check_leucine_module(ko_list):
    status = {}
    status["Isopropylmalate_synthase"] = ("K01649" in ko_list)
    status["Isopropylmalate_isomerase"] = (
        "K01702" in ko_list or ("K01703" in ko_list and "K01704" in ko_list)
    )
    status["Isopropylmalate_dehydrogenase"] = ("K00052" in ko_list)
    complete = all(status.values())
    return complete, status

# === STEP 5: Build KEGG-accurate leucine pathway model ===
def build_leucine_model():
    model = Model("Leucine_pathway")

    # Define compartments
    model.compartments = {"c": "cytosol", "e": "extracellular"}

    # Cytosolic metabolites
    oxoisovalerate_c = Metabolite("C00141_oxoisovalerate_c", compartment="c")
    acetylcoa_c = Metabolite("C00024_acetylcoa_c", compartment="c")
    isopropylmalate_c = Metabolite("C02504_isopropylmalate_c", compartment="c")
    isopropylmalate3_c = Metabolite("C04411_isopropylmalate3_c", compartment="c")
    isopropylmaleate_c = Metabolite("C02631_isopropylmaleate_c", compartment="c")
    oxoisocaproate_c = Metabolite("C00233_oxoisocaproate_c", compartment="c")
    oxosuccinate_c = Metabolite("C04236_oxosuccinate_c", compartment="c")
    coa_c = Metabolite("C00010_coa_c", compartment="c")
    h2o_c = Metabolite("C00001_h2o_c", compartment="c")
    nad_c = Metabolite("C00003_nad_c", compartment="c")
    nadh_c = Metabolite("C00004_nadh_c", compartment="c")
    h_c = Metabolite("C00080_h_c", compartment="c")
    co2_c = Metabolite("C00011_co2_c", compartment="c")

    # Reactions
    r01213 = Reaction("R01213_Isopropylmalate_synthase")
    r01213.add_metabolites({
        isopropylmalate_c:-1,
        coa_c:-1,
        acetylcoa_c:1,
        oxoisovalerate_c:1,
        h2o_c:1
    })

    r03968 = Reaction("R03968_Isopropylmalate_hydrolyase")
    r03968.add_metabolites({
        isopropylmalate_c:-1,
        isopropylmaleate_c:1,
        h2o_c:1
    })

    r04001 = Reaction("R04001_3Isopropylmalate_hydrolyase")
    r04001.add_metabolites({
        isopropylmalate3_c:-1,
        isopropylmaleate_c:1,
        h2o_c:1
    })

    r04426 = Reaction("R04426_Isopropylmalate_dehydrogenase")
    r04426.add_metabolites({
        isopropylmalate3_c:-1,
        nad_c:-1,
        oxosuccinate_c:1,
        nadh_c:1,
        h_c:1
    })

    r01652 = Reaction("R01652_Oxosuccinate_to_oxoisocaproate")
    r01652.add_metabolites({
        oxosuccinate_c:-1,
        oxoisocaproate_c:1,
        co2_c:1
    })

    model.add_reactions([r01213, r03968, r04001, r04426, r01652])

    # External metabolites
    oxoisovalerate_e = Metabolite("C00141_oxoisovalerate_e", compartment="e")
    acetylcoa_e = Metabolite("C00024_acetylcoa_e", compartment="e")
    nad_e = Metabolite("C00003_nad_e", compartment="e")
    coa_e = Metabolite("C00010_coa_e", compartment="e")
    h2o_e = Metabolite("C00001_h2o_e", compartment="e")

    # Exchanges linking external to cytosol
    ex1 = Reaction("EX_C00141")
    ex1.add_metabolites({oxoisovalerate_e:-1, oxoisovalerate_c:1})
    ex1.lower_bound = -1000; ex1.upper_bound = 1000

    ex2 = Reaction("EX_C00024")
    ex2.add_metabolites({acetylcoa_e:-1, acetylcoa_c:1})
    ex2.lower_bound = -1000; ex2.upper_bound = 1000

    ex3 = Reaction("EX_C00003")
    ex3.add_metabolites({nad_e:-1, nad_c:1})
    ex3.lower_bound = -1000; ex3.upper_bound = 1000

    ex4 = Reaction("EX_C00010")
    ex4.add_metabolites({coa_e:-1, coa_c:1})
    ex4.lower_bound = -1000; ex4.upper_bound = 1000

    ex5 = Reaction("EX_C00001")
    ex5.add_metabolites({h2o_e:-1, h2o_c:1})
    ex5.lower_bound = -1000; ex5.upper_bound = 1000

    model.add_reactions([ex1, ex2, ex3, ex4, ex5])

    # Demand reaction for C00233
    dm_leu = Reaction("DM_C00233_oxoisocaproate_c")
    dm_leu.add_metabolites({oxoisocaproate_c:-1})
    dm_leu.lower_bound = 0
    dm_leu.upper_bound = 1000
    model.add_reactions([dm_leu])
    model.objective = dm_leu

    # Medium fix
    model.medium = {
        "EX_C00141": 1000,
        "EX_C00024": 1000,
        "EX_C00003": 1000,
        "EX_C00010": 1000,
        "EX_C00001": 1000
    }

    return model

# === STEP 6: Run genome-wise analysis ===
summary = []
for genome, kos in genome_kos.items():
    complete, status = check_leucine_module(kos)
    flux_value = None
    if complete:
        model = build_leucine_model()
        sol = model.optimize()
        flux_value = sol.objective_value
    summary.append({
        "Genome": genome,
        "Isopropylmalate_synthase": status["Isopropylmalate_synthase"],
        "Isopropylmalate_isomerase": status["Isopropylmalate_isomerase"],
        "Isopropylmalate_dehydrogenase": status["Isopropylmalate_dehydrogenase"],
        "Module_complete": complete,
        "Flux_to_C00233": flux_value
    })

# === STEP 7: Output results ===
df = pd.DataFrame(summary)
print(df)
df.to_csv("leucine_flux_genomewise.tsv", sep="\t", index=False)

# === STEP 8: Download TSV ===
files.download("leucine_flux_genomewise.tsv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.8/141.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 84.4 MB/s eta 0:00:00


Saving FINAL BLAST.tsv to FINAL BLAST.tsv
    Genome  Isopropylmalate_synthase  Isopropylmalate_isomerase  \
0    ADCAI                      True                       True   
1    AFCAI                      True                       True   
2      BTB                      True                       True   
3      BTQ                      True                       True   
4   BTQVLC                      True                       True   
5     BTZ1                      True                       True   
6    China                      True                       True   
7   China1                      True                       True   
8    MEAM1                      True                       True   
9     PeMo                      True                       True   
10    SiSi                      True                       True   
11      TV                      True                       True   

    Isopropylmalate_dehydrogenase  Module_complete  Flux_to_C00233  
0                

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# === Unified Diagnostic for Leucine Pathway ===

import cobra

# Build the leucine model again
model = build_leucine_model()

print("\n=== OBJECTIVE ===")
print(model.objective.expression)

# 1. Exchange reactions
print("\n=== EXCHANGE REACTIONS ===")
for rxn in model.exchanges:
    print(rxn.id, rxn.reaction, rxn.lower_bound, rxn.upper_bound)

# 2. Medium availability
print("\n=== MEDIUM ===")
print(model.medium)

# 3. Metabolite connectivity
print("\n=== METABOLITE CONNECTIVITY ===")
for met in model.metabolites:
    print(met.id, [r.id for r in met.reactions])

# 4. Blocked reactions
print("\n=== BLOCKED REACTIONS ===")
blocked = cobra.flux_analysis.find_blocked_reactions(model)
print(blocked)

# 5. Flux distribution
print("\n=== FLUX DISTRIBUTION ===")
solution = model.optimize()
print(solution.fluxes)

# 6. Model summary
print("\n=== MODEL SUMMARY ===")
print(model.summary())

# 7. Check demand reaction directly
print("\n=== DEMAND REACTION ===")
dm_rxn = model.reactions.get_by_id("DM_C00233_oxoisocaproate_c")
print("Demand bounds:", dm_rxn.lower_bound, dm_rxn.upper_bound)
print("Demand metabolites:", dm_rxn.metabolites)



=== OBJECTIVE ===
1.0*DM_C00233_oxoisocaproate_c - 1.0*DM_C00233_oxoisocaproate_c_reverse_f4dda

=== EXCHANGE REACTIONS ===

=== MEDIUM ===
{}

=== METABOLITE CONNECTIVITY ===
C02504_isopropylmalate_c ['R01213_Isopropylmalate_synthase', 'R03968_Isopropylmalate_hydrolyase']
C00010_coa_c ['R01213_Isopropylmalate_synthase', 'EX_C00010']
C00024_acetylcoa_c ['R01213_Isopropylmalate_synthase', 'EX_C00024']
C00141_oxoisovalerate_c ['R01213_Isopropylmalate_synthase', 'EX_C00141']
C00001_h2o_c ['R01213_Isopropylmalate_synthase', 'R04001_3Isopropylmalate_hydrolyase', 'R03968_Isopropylmalate_hydrolyase', 'EX_C00001']
C02631_isopropylmaleate_c ['R04001_3Isopropylmalate_hydrolyase', 'R03968_Isopropylmalate_hydrolyase']
C04411_isopropylmalate3_c ['R04001_3Isopropylmalate_hydrolyase', 'R04426_Isopropylmalate_dehydrogenase']
C00003_nad_c ['EX_C00003', 'R04426_Isopropylmalate_dehydrogenase']
C04236_oxosuccinate_c ['R04426_Isopropylmalate_dehydrogenase', 'R01652_Oxosuccinate_to_oxoisocaproate']
C00004_

In [ ]:
# === STEP 1: Install dependencies ===
!pip install cobra pandas

import cobra
from cobra import Model, Reaction, Metabolite
import pandas as pd
from google.colab import files

# === STEP 2: Upload KO annotation file ===
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# === STEP 3: Parse KO annotations genome-wise ===
genome_kos = {}
with open(filename, "r") as f:
    for line in f:
        if not line.strip(): continue
        parts = line.strip().split("\t")
        if len(parts) < 2: continue
        genome_id, ko = parts[0], parts[1]
        genome_prefix = genome_id.split("_")[0]
        genome_kos.setdefault(genome_prefix, set()).add(ko)

# === STEP 4: KO logic for leucine biosynthesis up to 2-oxoisovalerate ===
def check_leucine_module(ko_list):
    status = {}
    status["Acetolactate_synthase"] = any(k in ko_list for k in ["K01649","K01650","K01651"])
    status["Acetohydroxyacid_isomeroreductase"] = ("K00052" in ko_list)
    status["Dihydroxyacid_dehydratase"] = ("K01687" in ko_list)
    complete = all(status.values())
    return complete, status

# === STEP 5: Build leucine pathway model (up to 2-oxoisovalerate) ===
def build_leucine_model():
    model = Model("Leucine_pathway")

    # Metabolites
    pyr = Metabolite("C00022_c", compartment="c")        # Pyruvate
    acetolactate = Metabolite("C00118_c", compartment="c") # 2-Acetolactate
    dhiv = Metabolite("C01179_c", compartment="c")       # 2,3-Dihydroxyisovalerate
    oxoisovalerate = Metabolite("C00183_c", compartment="c") # 2-Oxoisovalerate
    nadph = Metabolite("C00005_c", compartment="c")
    nadp = Metabolite("C00006_c", compartment="c")
    h2o = Metabolite("C00001_c", compartment="c")
    h = Metabolite("C00080_c", compartment="c")

    # Reactions
    r00200 = Reaction("R00200_Acetolactate_synthase")
    r00200.add_metabolites({pyr:-2, acetolactate:1})

    r02740 = Reaction("R02740_Acetohydroxyacid_isomeroreductase")
    r02740.add_metabolites({acetolactate:-1, nadph:-1, h2o:-1,
                            dhiv:1, nadp:1, h:1})

    r01714 = Reaction("R01714_Dihydroxyacid_dehydratase")
    r01714.add_metabolites({dhiv:-1, oxoisovalerate:1, h2o:1})

    model.add_reactions([r00200, r02740, r01714])

    # Exchanges for all metabolites
    for met in [pyr, acetolactate, dhiv, oxoisovalerate, nadph, nadp, h2o, h]:
        ex = Reaction(f"EX_{met.id}")
        ex.add_metabolites({met:1})
        ex.lower_bound = -1000
        ex.upper_bound = 1000
        model.add_reactions([ex])

    # Demand reaction for 2-oxoisovalerate
    dm_leu_precursor = Reaction("DM_C00183")
    dm_leu_precursor.add_metabolites({oxoisovalerate:-1})
    dm_leu_precursor.lower_bound = 0
    dm_leu_precursor.upper_bound = 1000
    model.add_reactions([dm_leu_precursor])
    model.objective = dm_leu_precursor

    return model

# === STEP 6: Run genome-wise analysis ===
summary = []
for genome, kos in genome_kos.items():
    complete, status = check_leucine_module(kos)
    flux_value = None
    if complete:
        model = build_leucine_model()
        sol = model.optimize()
        flux_value = sol.objective_value
    summary.append({
        "Genome": genome,
        **status,
        "Module_complete": complete,
        "Flux_to_2Oxoisovalerate": flux_value
    })

# === STEP 7: Output results ===
df = pd.DataFrame(summary)
print(df)
df.to_csv("leucine_flux_genomewise.tsv", sep="\t", index=False)

# === STEP 8: Download TSV ===
files.download("leucine_flux_genomewise.tsv")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.8/141.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 55.8 MB/s eta 0:00:00


Saving FINAL BLAST.tsv to FINAL BLAST.tsv
    Genome  Acetolactate_synthase  Acetohydroxyacid_isomeroreductase  \
0    ADCAI                   True                               True   
1    AFCAI                   True                               True   
2      BTB                   True                               True   
3      BTQ                   True                               True   
4   BTQVLC                   True                               True   
5     BTZ1                   True                               True   
6    China                   True                              False   
7   China1                   True                               True   
8    MEAM1                   True                               True   
9     PeMo                   True                               True   
10    SiSi                   True                               True   
11      TV                   True                               True   

    Dihydroxyacid_deh

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>